# Notebook 1 - Chatbot vs. Agent

## Why this notebook exists

You have used a chatbot. You type a question, you get an answer, you type again. It never *does* anything on its own.

An agent is different: it can break a task into steps, decide to use a tool, look at the result, and keep going until it is actually done. That difference is the whole workshop. Everything else (tools, retrieval, memory) hangs off it.

By the end of this notebook you will be able to **explain why an agent behaves differently from a chatbot**, because you will have built both and watched them run on the same task, side by side.

## What you'll build
1. A **chatbot**: one prompt in, one answer out.
2. An **agent**: the same task, but running a **plan -> act -> observe** loop you can read line by line.
3. A **side-by-side comparison** so the difference is visible, not asserted.

## Prerequisites
- A Google account (For Google Colab, no installation).
- Optional: an Anthropic **or** OpenAI API key. You don't need one to start. The notebook runs in `"mock"` mode with canned replies so you can see the mechanics first, then switch to a real model.

One term, used consistently all workshop: an **agent** is a language model wrapped in a loop that lets it call tools and decide when it is finished.

## Step 1 - Setup

Run the cell below. Read it. This same adapter appears in every notebook. It is the seam that lets you swap Anthropic for OpenAI by changing one word.

In [ ]:
#  SETUP cell. This is for a provider-agnostic LLM client.
#  Read this cell; every notebook uses this same adapter.
#  Switch providers by changing PROVIDER.
#  Nothingelse in the notebook changes. an agent is a pattern, not a vendor.

# In Google Colab this cell installs the packages. Locally, run once.
try:
    !pip -q install anthropic openai
except Exception:
    pass

import os, json

PROVIDER = "openai"   # "anthropic" | "openai" | "mock"
                      # "mock" runs this notebook with NO API key, using canned
                      # replies, so you can watch the mechanics before you spend
                      # a cent. Switch to a real provider once it makes sense.

MODELS = {
    "anthropic": "claude-haiku-4-5-20251001",  # cheapest current Claude model
    "openai":    "gpt-5.4-nano", # cheapest current GPT model
}

# API keys
# In Colab: click the key icon (left sidebar) -> add ANTHROPIC_API_KEY or
# OPENAI_API_KEY, then uncomment the two lines below to load them.
#from google.colab import userdata
# os.environ["ANTHROPIC_API_KEY"] = userdata.get("ANTHROPIC_API_KEY")
#os.environ["OPENAI_API_KEY"] = userdata.get("MY_API_KEY")

# Normalized shapes we use everywhere (so the agent code never mentions a vendor):
#   message: {"role":"user","content":str}
#            {"role":"assistant","content":str|None,"tool_calls":[{id,name,args}]}
#            {"role":"tool","tool_call_id":id,"name":name,"content":str}
#   tool:    {"name":str,"description":str,"parameters":<json-schema>}
#   reply:   {"text":str,"tool_calls":[{id,name,args}],"stop_reason":str}


_MOCK_SCRIPT = []
def mock_reset(script):
    global _MOCK_SCRIPT; _MOCK_SCRIPT = list(script)
def _mock_call(messages, tools):
    if _MOCK_SCRIPT:
        kind, payload = _MOCK_SCRIPT.pop(0)
        if kind == "tool":
            return {"text":"", "tool_calls":[{"id":"mock_"+payload["name"],
                    "name":payload["name"], "args":payload["args"]}], "stop_reason":"tool_use"}
        return {"text":payload, "tool_calls":[], "stop_reason":"end"}
    last = next((m for m in reversed(messages) if m["role"] in ("user","tool")), {"content":""})
    return {"text": f"[mock reply to] {str(last.get('content',''))[:80]}",
            "tool_calls":[], "stop_reason":"end"}


def _to_anthropic(messages):
    out=[]
    for m in messages:
        if m["role"]=="user":
            out.append({"role":"user","content":m["content"]})
        elif m["role"]=="assistant":
            blocks=[]
            if m.get("content"): blocks.append({"type":"text","text":m["content"]})
            for tc in m.get("tool_calls",[]):
                blocks.append({"type":"tool_use","id":tc["id"],"name":tc["name"],"input":tc["args"]})
            out.append({"role":"assistant","content":blocks})
        elif m["role"]=="tool":
            out.append({"role":"user","content":[{"type":"tool_result",
                        "tool_use_id":m["tool_call_id"],"content":str(m["content"])}]})
    return out

def _to_openai(messages, system):
    out=[]
    if system: out.append({"role":"system","content":system})
    for m in messages:
        if m["role"]=="user":
            out.append({"role":"user","content":m["content"]})
        elif m["role"]=="assistant":
            msg={"role":"assistant","content":m.get("content") or None}
            if m.get("tool_calls"):
                msg["tool_calls"]=[{"id":tc["id"],"type":"function","function":{
                    "name":tc["name"],"arguments":json.dumps(tc["args"])}} for tc in m["tool_calls"]]
            out.append(msg)
        elif m["role"]=="tool":
            out.append({"role":"tool","tool_call_id":m["tool_call_id"],"content":str(m["content"])})
    return out

def call_llm(messages, tools=None, system=None, max_tokens=1024, temperature=0):
    '''One function. Any provider. This is portable across whatever API key you have or your institution has.'''
    if PROVIDER=="mock":
        return _mock_call(messages, tools)
    if PROVIDER=="anthropic":
        from anthropic import Anthropic
        client=Anthropic()
        kw=dict(model=MODELS["anthropic"], max_tokens=max_tokens,
                temperature=temperature, messages=_to_anthropic(messages))
        if system: kw["system"]=system
        if tools: kw["tools"]=[{"name":t["name"],"description":t["description"],
                                "input_schema":t["parameters"]} for t in tools]
        r=client.messages.create(**kw)
        text=""; calls=[]
        for b in r.content:
            if b.type=="text": text+=b.text
            elif b.type=="tool_use": calls.append({"id":b.id,"name":b.name,"args":b.input})
        return {"text":text,"tool_calls":calls,"stop_reason":r.stop_reason}
    if PROVIDER=="openai":
        from openai import OpenAI
        client=OpenAI()
        kw=dict(model=MODELS["openai"], messages=_to_openai(messages, system),
                temperature=temperature, max_completion_tokens=max_tokens)
        if tools: kw["tools"]=[{"type":"function","function":{"name":t["name"],
                    "description":t["description"],"parameters":t["parameters"]}} for t in tools]
        r=client.chat.completions.create(**kw)
        msg=r.choices[0].message
        calls=[{"id":tc.id,"name":tc.function.name,
                "args":json.loads(tc.function.arguments or "{}")} for tc in (msg.tool_calls or [])]
        return {"text":msg.content or "", "tool_calls":calls,
                "stop_reason":"tool_use" if calls else "end"}
    raise ValueError("Unknown PROVIDER: "+PROVIDER)

print(f"Setup ready. PROVIDER={PROVIDER!r}. Switch to 'anthropic' or 'openai' when you add a key.")

zsh:1: command not found: pip
Setup ready. PROVIDER='openai'. Switch to 'anthropic' or 'openai' when you add a key.


## Step 2 — The chatbot

A chatbot is a **single call** to a language model. Prompt goes in, text comes out, and the model has no way to take an action in the world.

Under the hood the model generates one token at a time, each conditioned on everything before it.

That is powerful, but it is *one shot*. If the task needs a real fact the model doesn't have, or a calculation it can't do reliably in its head, a chatbot will guess. Watch it do exactly that.

In [ ]:
def chatbot(prompt, system=None):
    '''One call. One answer. No tools, no loop.'''
    reply = call_llm([{"role":"user","content":prompt}], system=system)
    return reply["text"]

# A task with a fact the model cannot know reliably: today's date, then arithmetic on it.
task = "How many days are there between today's date and the start of 2015? Show the number."

# Uncomment this part if you're executing on mock provider mode
'''
mock_reset([("final",
    "There are roughly 4,000 days between 2015 and now. (A single-shot guess: "
    "I have no way to check today's real date, so this may be wrong.)")])
PROVIDER = "mock"
'''

print("CHATBOT answer:\n", chatbot(task))

CHATBOT answer:
 Today is **2026-07-14**. The start of **2015-01-01** is **2,?** days earlier.

**Answer: 2,? days**


The chatbot answered in one breath. If it doesn't actually know today's date, it *guesses* and moves on. There is no step where it could go find out.

That is the gap an agent closes.

## Step 3 — The agent loop

An agent is the same model, but wrapped in a loop and handed some tools. Each turn it can either **call a tool** or **give a final answer**. When it calls a tool, we run the real function, feed the result back, and let it continue.

This pattern is called **ReAct** (Reason + Act): the model reasons about what to do, acts by calling a tool, observes the result, and repeats.

user task --> model
if (no tool call) --> final answer
if (tool call) --> run the tool --> observe (feed result back) --> loop again


In [ ]:
def run_agent(user_task, tools_spec, tool_fns, system=None, max_steps=6, verbose=True):
    '''The agent loop. The idea is:
       plan -> act (maybe call a tool) -> observe (feed the result back) -> repeat,
       until the model returns a final answer instead of a tool call.'''
    messages = [{"role":"user","content":user_task}]
    for step in range(1, max_steps+1):
        reply = call_llm(messages, tools=tools_spec, system=system)
        if reply["tool_calls"]:                                  # the model wants a tool
            messages.append({"role":"assistant","content":reply["text"],
                             "tool_calls":reply["tool_calls"]})
            for tc in reply["tool_calls"]:
                if verbose: print(f"  step {step}: tool `{tc['name']}` <- {tc['args']}")
                try:    result = tool_fns[tc["name"]](**tc["args"])   # run the real function
                except Exception as e: result = f"ERROR: {e}"
                messages.append({"role":"tool","tool_call_id":tc["id"],
                                 "name":tc["name"],"content":result})   # observe
        else:                                                    # no tool -> we are done
            if verbose: print(f"  step {step}: final answer")
            return reply["text"], messages
    return "Stopped: hit max_steps.", messages

### Give the agent two real tools

A **tool** is just a Python function plus a small description that tells the model the tool exists and what arguments it takes. Here are two: one to get today's real date, one to do arithmetic exactly (models are unreliable at mental math; a calculator tool fixes that).

In [ ]:
from datetime import date

def today():
    '''Returning today's date in ISO format.'''
    return date.today().isoformat()

def days_between(start, end):
    '''Whole days between two dates (end - start).'''
    from datetime import date as d
    a = d.fromisoformat(start); b = d.fromisoformat(end)
    return str((b - a).days)

# The schemas the MODEL sees. The model chooses a tool by reading the descriptions.
# Vague descriptions cause wrong tool choices.
tools_spec = [
    {"name":"today", "description":"Get today's real calendar date (ISO YYYY-MM-DD).",
     "parameters":{"type":"object","properties":{}, "required":[]}},
    {"name":"days_between",
     "description":"Number of whole days between two ISO dates, computed as end minus start.",
     "parameters":{"type":"object","properties":{
        "start":{"type":"string","description":"ISO date YYYY-MM-DD"},
        "end":{"type":"string","description":"ISO date YYYY-MM-DD"}},
        "required":["start","end"]}},
]
tool_fns = {"today": today, "days_between": days_between}
print("2 tools registered:", list(tool_fns))

2 tools registered: ['today', 'days_between']


In [ ]:
# Script the agent's decisions for mock mode: get the date, then compute, then answer.
# (With a real provider, the MODEL makes these decisions itself. Set PROVIDER above.)
real_today = today()
mock_reset([
    ("tool", {"name":"today", "args":{}}),
    ("tool", {"name":"days_between", "args":{"start":"2015-01-01", "end":real_today}}),
    ("final", f"Today is {real_today}. Days since 2015-01-01: computed exactly by the tool above."),
])

print("AGENT run:")
answer, transcript = run_agent(
    "How many days between today and the start of 2015? Use tools; do not guess.",
    tools_spec, tool_fns)
print("\nAGENT answer:\n", answer)

AGENT run:
  step 1: tool `today` <- {}
  step 2: tool `days_between` <- {'start': '2015-01-01', 'end': '2026-07-14'}
  step 3: final answer

AGENT answer:
 There are **4,212 days** between **today (2026-07-14)** and the start of **2015 (2015-01-01)**.


## Step 4 — Side by side

Notice the difference in *behaviour*, not just wording.

In [ ]:
print("="*68)
print("CHATBOT  : one call, no way to check facts -> guesses.")
print("AGENT    : loops, calls `today`, then `days_between` -> exact answer.")
print("="*68)
print("\nThe agent's transcript (what actually happened each turn):")
for m in transcript:
    role = m["role"]
    if role == "assistant" and m.get("tool_calls"):
        for tc in m["tool_calls"]:
            print(f"  assistant -> call {tc['name']}({tc['args']})")
    elif role == "tool":
        print(f"  tool[{m['name']}] -> {m['content']}")
    elif role == "user":
        print(f"  user -> {m['content']}")

CHATBOT  : one call, no way to check facts -> guesses.
AGENT    : loops, calls `today`, then `days_between` -> exact answer.

The agent's transcript (what actually happened each turn):
  user -> How many days between today and the start of 2015? Use tools; do not guess.
  assistant -> call today({})
  tool[today] -> 2026-07-14
  assistant -> call days_between({'start': '2015-01-01', 'end': '2026-07-14'})
  tool[days_between] -> 4212


## The anatomy of an agent

Every agent you will ever meet is some arrangement of these five parts. You just built the first three.

- **A model** - the language model that reasons and decides. (the `call_llm` seam)
- **A loop** - turns one answer into many steps. (`run_agent`)
- **Tools** - functions that let it act in the world. (Notebook 2 goes deep)
- **Memory** - context kept across steps. (Notebook 4)
- **A stopping condition** - how it knows it's done. (here: "no tool call = done", plus `max_steps` as a safety net)


## Exercise

Give the agent a **two-step question of your own** and watch it loop. A good shape for a question is *"find X, then use X to compute Y."*

1. In the cell below, write your task.
2. If you are still in `"mock"` mode, script the tool calls the way we did above. If you've added a real key and set `PROVIDER="anthropic"` or `"openai"`, delete the `mock_reset(...)` line and let the model decide.
3. Run it and read the transcript.

**Question to answer for yourself:** at which step did the agent do something a chatbot *could not*?

In [ ]:
# YOUR TASK
my_task = "Find today's date, then tell me how many days until the end of this year."

# Mock script (Once you set a real PROVIDER, rempve this):
mock_reset([
    ("tool", {"name":"today", "args":{}}),
    ("tool", {"name":"days_between", "args":{"start":today(), "end":"2026-12-31"}}),
    ("final", "See the tool result above for the exact number of days remaining this year."),
])

ans, tr = run_agent(my_task, tools_spec, tool_fns)
print("\nAnswer:", ans)

  step 1: tool `today` <- {}
  step 2: tool `days_between` <- {'start': '2026-07-14', 'end': '2026-12-31'}
  step 3: final answer

Answer: Today’s date is **2026-07-14**.  
There are **170 days** until the end of this year (**2026-12-31**).


## Workflows vs. agents

- **Workflow:** you, the developer, fix the steps in advance (call model, then search, then summarize). Predictable, cheaper, easier to debug.
- **Agent:** the *model* decides the steps at run time. Flexible, but more calls, more cost, harder to predict.

**Cost is the honest catch.** A chatbot is one API call. An agent that takes 4 tool-steps is at least 5 calls, and each call resends the growing transcript. So an agent can cost several times what a chatbot costs for the same question. Use the loop where the flexibility earns its price, not everywhere.

In [ ]:
# Rough cost intuition (not a benchmark): count model calls in the run above.
calls = sum(1 for m in transcript if m["role"]=="assistant")
print(f"This agent run made ~{calls}+ model calls vs. 1 for the chatbot.")
print("Rule of thumb: reach for the loop when the steps can't be known in advance.")

This agent run made ~2+ model calls vs. 1 for the chatbot.
Rule of thumb: reach for the loop when the steps can't be known in advance.


## Reflect and Discuss

- When is a **chatbot** enough? (single, self-contained question; no external fact or action needed)
- When do you actually need the **loop**? (multi-step, needs a real fact, needs to *do* something)
- Where in *your* research could a step "go find out and come back" save you time?